In [2]:
# Connecting to MongoDB

from pyspark.sql import SparkSession

# Creating Spark Session
spark = SparkSession.builder \
    .appName("PlaystoreAnalysis") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()


In [3]:

# Reading the data from MondoDB
df = spark.read.format("mongodb") \
    .option("database", "playstore") \
    .option("collection", "apps") \
    .load()

# Confirming if the data reading succeeded
df.printSchema()
df.show()

root
 |-- Ad Supported: boolean (nullable = true)
 |-- App Id: string (nullable = true)
 |-- App Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Editors Choice: boolean (nullable = true)
 |-- Free: boolean (nullable = true)
 |-- In App Purchases: boolean (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Last Updated: timestamp (nullable = true)
 |-- Maximum Installs: long (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Minimum Installs: long (nullable = true)
 |-- Price: double (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Rating Count: integer (nullable = true)
 |-- Released: timestamp (nullable = true)
 |-- Size: double (nullable = true)
 |-- _id: string (nullable = true)

+------------+--------------------+----------------------------------+-----------------+--------------+-------

## Data Analysis and Visualization

### Dependency analysis of apps ratings and price type

In [ ]:
from pyspark.sql.functions import col, avg, stddev, count, when, round as spark_round
import pandas as pd
import matplotlib.pyplot as plt

df.createOrReplaceTempView("apps")

# ============================================================
# ANALYSIS 1: BASIC STATISTICS - FREE VS PAID RATINGS
# ============================================================

print("="*60)
print("RATING STATISTICS: FREE VS PAID APPS")
print("="*60)

rating_stats = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        COUNT(*) AS total_apps_with_ratings,
        ROUND(AVG(Rating), 2) AS avg_rating,
        ROUND(STDDEV(Rating), 2) AS stddev_rating,
        ROUND(MIN(Rating), 2) AS min_rating,
        ROUND(MAX(Rating), 2) AS max_rating,
        ROUND(PERCENTILE_APPROX(Rating, 0.5), 2) AS median_rating
    FROM apps
    WHERE Rating IS NOT NULL AND Rating > 0
    GROUP BY Free
""")

rating_stats.show(truncate=False)

# ============================================================
# ANALYSIS 2: RATING DISTRIBUTION BY PRICE TYPE
# ============================================================

print("\n" + "="*60)
print("RATING DISTRIBUTION: FREE VS PAID")
print("="*60)

rating_distribution = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        CASE 
            WHEN Rating >= 4.5 THEN '4.5 - 5.0 (Excellent)'
            WHEN Rating >= 4.0 THEN '4.0 - 4.5 (Good)'
            WHEN Rating >= 3.5 THEN '3.5 - 4.0 (Average)'
            WHEN Rating >= 3.0 THEN '3.0 - 3.5 (Below Average)'
            WHEN Rating >= 2.0 THEN '2.0 - 3.0 (Poor)'
            ELSE 'Below 2.0 (Very Poor)'
        END AS rating_category,
        COUNT(*) AS app_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY Free), 2) AS percentage
    FROM apps
    WHERE Rating IS NOT NULL AND Rating > 0
    GROUP BY Free, rating_category
    ORDER BY Free DESC, rating_category
""")

rating_distribution.show(20, truncate=False)

# ============================================================
# ANALYSIS 3: STATISTICAL SIGNIFICANCE
# ============================================================

print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE TEST")
print("="*60)

significance = spark.sql("""
    WITH stats AS (
        SELECT 
            Free,
            COUNT(*) AS n,
            AVG(Rating) AS mean,
            STDDEV(Rating) AS stddev,
            VARIANCE(Rating) AS variance
        FROM apps
        WHERE Rating IS NOT NULL AND Rating > 0
        GROUP BY Free
    )
    SELECT 
        MAX(CASE WHEN Free = true THEN mean END) AS free_mean,
        MAX(CASE WHEN Free = false THEN mean END) AS paid_mean,
        MAX(CASE WHEN Free = true THEN stddev END) AS free_stddev,
        MAX(CASE WHEN Free = false THEN stddev END) AS paid_stddev,
        MAX(CASE WHEN Free = true THEN n END) AS free_n,
        MAX(CASE WHEN Free = false THEN n END) AS paid_n,
        ABS(MAX(CASE WHEN Free = true THEN mean END) - MAX(CASE WHEN Free = false THEN mean END)) AS mean_difference
    FROM stats
""").collect()[0]

print(f"Free apps average rating: {significance['free_mean']:.2f}")
print(f"Paid apps average rating: {significance['paid_mean']:.2f}")
print(f"Difference: {significance['mean_difference']:.2f}")

# Calculate effect size (Cohen's d approximation)
pooled_std = ((significance['free_stddev']**2 + significance['paid_stddev']**2) / 2) ** 0.5
effect_size = significance['mean_difference'] / pooled_std

print(f"\nEffect size (Cohen's d): {effect_size:.3f}")
if abs(effect_size) < 0.2:
    print("Interpretation: Negligible effect")
elif abs(effect_size) < 0.5:
    print("Interpretation: Small effect")
elif abs(effect_size) < 0.8:
    print("Interpretation: Medium effect")
else:
    print("Interpretation: Large effect")

# ============================================================
# ANALYSIS 4: TOP RATED APPS BY PRICE TYPE
# ============================================================

print("\n" + "="*60)
print("TOP 5 HIGHEST RATED FREE APPS")
print("="*60)

top_free = spark.sql("""
    SELECT 
        `App Name` AS app_name,
        Category,
        ROUND(Rating, 2) AS rating,
        `Rating Count` AS review_count
    FROM apps
    WHERE Free = true 
        AND Rating IS NOT NULL 
        AND Rating > 0
    ORDER BY Rating DESC
    LIMIT 5
""")
top_free.show(truncate=False)

print("\n" + "="*60)
print("TOP 5 HIGHEST RATED PAID APPS")
print("="*60)

top_paid = spark.sql("""
    SELECT 
        `App Name` AS app_name,
        Category,
        ROUND(Rating, 2) AS rating,
        `Rating Count` AS review_count,
        CONCAT('$', Price) AS price
    FROM apps
    WHERE Free = false 
        AND Rating IS NOT NULL 
        AND Rating > 0
    ORDER BY Rating DESC
    LIMIT 5
""")
top_paid.show(truncate=False)

# ============================================================
# ANALYSIS 5: SUMMARY AND CONCLUSION
# ============================================================

print("\n" + "="*60)
print("SUMMARY AND CONCLUSIONS")
print("="*60)

summary = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        COUNT(*) AS total_apps_with_real_ratings,
        ROUND(AVG(Rating), 2) AS avg_rating,
        ROUND(PERCENTILE_APPROX(Rating, 0.5), 2) AS median_rating,
        ROUND(STDDEV(Rating), 3) AS rating_stddev,
        COUNT(CASE WHEN Rating >= 4.0 THEN 1 END) AS high_rated_apps,
        ROUND(COUNT(CASE WHEN Rating >= 4.0 THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_high_rated
    FROM apps
    WHERE Rating IS NOT NULL AND Rating > 0
    GROUP BY Free
""")

summary.show(truncate=False)

# Statistical conclusion
free_avg = summary.filter(col("price_type") == "Free").select("avg_rating").collect()[0][0]
paid_avg = summary.filter(col("price_type") == "Paid").select("avg_rating").collect()[0][0]

print(f"\nKEY FINDINGS (REAL RATINGS ONLY):")
print(f"• Free apps average rating: {free_avg:.2f}")
print(f"• Paid apps average rating: {paid_avg:.2f}")
print(f"• Difference: {abs(free_avg - paid_avg):.2f} points")
print(f"• Effect size (Cohen's d): {effect_size:.3f}")

if effect_size < 0.2:
    print("\n• CONCLUSION: NO meaningful dependency between rating and price type")
    print("  The rating difference is negligible - less than 0.2 standard deviations")
elif effect_size < 0.5:
    print("\n• CONCLUSION: Weak dependency - Paid apps tend to have slightly higher ratings")
elif effect_size < 0.8:
    print("\n• CONCLUSION: Moderate dependency - Paid apps have noticeably higher ratings")
else:
    print("\n• CONCLUSION: Strong dependency - Paid apps have substantially higher ratings")

In [ ]:
# Spark table to Pandas
plot_data = rating_distribution.toPandas()

# Percentage to numeric
plot_data['percentage'] = pd.to_numeric(plot_data['percentage'])

x = plot_data.pivot(index='rating_category', columns='price_type', values='percentage')

# Ordering the categories logically
ordered_cats = [
        'Below 2.0 (Very Poor)', '2.0 - 3.0 (Poor)', 
        '3.0 - 3.5 (Below Average)', '3.5 - 4.0 (Average)', 
        '4.0 - 4.5 (Good)', '4.5 - 5.0 (Excellent)'
    ]

ax = x.reindex(ordered_cats)

# Plotting    
ax.plot(kind='barh', figsize=(10, 6))
plt.xlabel("Percentage (%)")
plt.show()